# 🔍 AML-мониторинг криптотранзакций: Детекция подозрительных транзакций Bitcoin
## RegTech | Надзор за цифровыми активами

**Датасет:** [Elliptic Bitcoin Dataset](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set)  
**Задача:** Бинарная классификация транзакций (легитимная / подозрительная)  
**Контекст:** Автоматизация AML-скрининга в соответствии с требованиями FATF Recommendation 16

---
### Структура проекта
1. Загрузка и первичный осмотр данных
2. EDA — разведочный анализ
3. Feature Engineering
4. Моделирование (Logistic Regression, Random Forest, XGBoost)
5. Оценка и сравнение моделей
6. Анализ стабильности данных (Data Drift)
7. Интерпретация результатов (SHAP)
8. Выводы и RegTech-импликации

## 0. Установка зависимостей

In [ ]:
# Установка необходимых библиотек
!pip install kaggle shap xgboost imbalanced-learn evidently matplotlib seaborn scikit-learn pandas numpy -q

## 1. Загрузка данных с Kaggle

**Инструкция:**
1. Зайдите на https://www.kaggle.com/settings → API → Create New Token → скачается `kaggle.json`
2. Поместите файл `kaggle.json` в папку `~/.kaggle/` (Linux/Mac) или `C:\Users\<user>\.kaggle\` (Windows)
3. Запустите ячейку ниже

In [ ]:
import os
import subprocess

# Скачать датасет с Kaggle
os.makedirs('data', exist_ok=True)

# Автоматическая загрузка через kaggle CLI
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'ellipticco/elliptic-data-set', '--unzip', '-p', 'data/'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print('Файлы в папке data/:', os.listdir('data/'))

In [ ]:
# Если kaggle CLI недоступен — скачайте вручную и положите файлы в папку data/
# Ожидаемые файлы:
# data/elliptic_bitcoin_dataset/elliptic_txs_features.csv
# data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv
# data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Попробуем найти файлы в разных местах
possible_paths = [
    'data/elliptic_bitcoin_dataset/',
    'data/',
    './'
]

data_path = None
for path in possible_paths:
    if os.path.exists(path + 'elliptic_txs_features.csv'):
        data_path = path
        break

if data_path:
    print(f'✅ Данные найдены в: {data_path}')
else:
    print('❌ Данные не найдены. Скачайте вручную с Kaggle и положите в папку data/')

## 2. Загрузка и первичный осмотр данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# ---- Загрузка данных ----
features_df = pd.read_csv(f'{data_path}elliptic_txs_features.csv', header=None)
classes_df  = pd.read_csv(f'{data_path}elliptic_txs_classes.csv')
edges_df    = pd.read_csv(f'{data_path}elliptic_txs_edgelist.csv')

# Именование колонок
# Колонка 0 = txId, колонка 1 = timestep, колонки 2-94 = локальные фичи, 95-166 = агрегированные
n_features = features_df.shape[1] - 2
local_feat_cols = [f'local_feat_{i}' for i in range(1, 94)]
agg_feat_cols   = [f'agg_feat_{i}'   for i in range(1, 73)]
features_df.columns = ['txId', 'timestep'] + local_feat_cols + agg_feat_cols

print(f'Features shape : {features_df.shape}')
print(f'Classes shape  : {classes_df.shape}')
print(f'Edges shape    : {edges_df.shape}')
print()
print('Распределение классов:')
print(classes_df['class'].value_counts())
print('  1 = illicit (подозрительная), 2 = licit (легитимная), unknown = неизвестна')

In [ ]:
# Объединяем features + classes
df = features_df.merge(classes_df, on='txId', how='left')

# Переводим метки в числа для обучения (только labeled данные)
df['label'] = df['class'].map({'1': 1, '2': 0, 1: 1, 2: 0})  # 1=illicit, 0=licit

# Размеченные данные
df_labeled = df[df['class'] != 'unknown'].copy()

print(f'Всего транзакций     : {len(df):,}')
print(f'Размеченных          : {len(df_labeled):,}')
print(f'Нелегитимных (illicit): {(df_labeled["label"]==1).sum():,} ({(df_labeled["label"]==1).mean()*100:.1f}%)')
print(f'Легитимных (licit)   : {(df_labeled["label"]==0).sum():,} ({(df_labeled["label"]==0).mean()*100:.1f}%)')
display(df_labeled.head(3))

## 3. EDA — Разведочный анализ данных

In [ ]:
# ---- 3.1 Дисбаланс классов ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Распределение транзакций', fontsize=14, fontweight='bold')

# Bar chart
class_counts = df_labeled['label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Легитимные (0)', 'Подозрительные (1)'], class_counts[[0,1]], color=colors, edgecolor='black', linewidth=0.7)
axes[0].set_title('Количество транзакций по классам')
axes[0].set_ylabel('Количество')
for i, v in enumerate(class_counts[[0,1]]):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie
axes[1].pie(class_counts[[0,1]], labels=['Легитимные', 'Подозрительные'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'black','linewidth':0.7})
axes[1].set_title('Доля классов')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Сильный дисбаланс классов — необходима балансировка при обучении!')

In [ ]:
# ---- 3.2 Транзакции по времени ----
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Временная динамика транзакций (49 временных шагов)', fontsize=14, fontweight='bold')

time_counts = df_labeled.groupby(['timestep', 'label']).size().unstack(fill_value=0)
time_counts.columns = ['Легитимные', 'Подозрительные']

# Абсолютные значения
time_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                 width=0.8, edgecolor='white', linewidth=0.3)
axes[0].set_title('Количество транзакций по временным шагам')
axes[0].set_xlabel('')
axes[0].set_ylabel('Количество')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=0)

# Доля подозрительных
illicit_rate = time_counts['Подозрительные'] / (time_counts['Легитимные'] + time_counts['Подозрительные'])
axes[1].plot(illicit_rate.index, illicit_rate.values * 100, color='#e74c3c', linewidth=2, marker='o', markersize=3)
axes[1].fill_between(illicit_rate.index, illicit_rate.values * 100, alpha=0.15, color='#e74c3c')
axes[1].set_title('Доля подозрительных транзакций по времени (%)')
axes[1].set_xlabel('Временной шаг')
axes[1].set_ylabel('%')
axes[1].axhline(illicit_rate.mean() * 100, color='gray', linestyle='--', label=f'Среднее: {illicit_rate.mean()*100:.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('temporal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 3.3 Качество данных ----
print('=== Качество данных ===')
print(f'Пропущенных значений: {df_labeled.isnull().sum().sum()}')
print(f'Дубликатов txId: {df_labeled["txId"].duplicated().sum()}')
print()

# Статистика фичей по классам
feat_cols = local_feat_cols + agg_feat_cols
stats = df_labeled.groupby('label')[feat_cols[:10]].agg(['mean', 'std'])
print('Статистика первых 10 признаков по классам:')
display(stats.T)

In [ ]:
# ---- 3.4 Корреляция топ-признаков с меткой ----
correlations = df_labeled[feat_cols + ['label']].corr()['label'].drop('label').abs().sort_values(ascending=False)
top_features = correlations.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v > 0.15 else '#3498db' for v in top_features.values]
ax.barh(range(len(top_features)), top_features.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index, fontsize=9)
ax.set_xlabel('|Корреляция с меткой (illicit)|')
ax.set_title('Топ-20 признаков по корреляции с целевой переменной')
ax.axvline(0.15, color='red', linestyle='--', alpha=0.5, label='Порог 0.15')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Топ-5 наиболее коррелированных признаков:')
for feat, val in top_features.head(5).items():
    print(f'  {feat}: {val:.4f}')

In [ ]:
# ---- 3.5 Распределение топ-признаков по классам ----
top5 = top_features.head(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Распределение топ-5 признаков по классам', fontsize=13, fontweight='bold')

for i, feat in enumerate(top5):
    data_licit   = df_labeled[df_labeled['label'] == 0][feat]
    data_illicit = df_labeled[df_labeled['label'] == 1][feat]
    
    # Clip outliers for visualization
    p99 = df_labeled[feat].quantile(0.99)
    p01 = df_labeled[feat].quantile(0.01)
    
    axes[i].hist(data_licit.clip(p01, p99), bins=40, alpha=0.6, color='#2ecc71', label='Легитимные', density=True)
    axes[i].hist(data_illicit.clip(p01, p99), bins=40, alpha=0.6, color='#e74c3c', label='Подозрительные', density=True)
    axes[i].set_title(feat, fontsize=8)
    axes[i].set_xlabel('Значение')
    if i == 0:
        axes[i].legend(fontsize=7)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 3.6 Тепловая карта корреляций топ-15 признаков ----
top15 = top_features.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df_labeled[top15].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Матрица корреляций топ-15 признаков', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering и подготовка данных

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ---- 4.1 Подготовка признаков ----
X = df_labeled[feat_cols].values
y = df_labeled['label'].values
timesteps = df_labeled['timestep'].values

# Временное разбиение: обучение на первых 34 шагах, тест на оставшихся 15
# Это реалистично для мониторинга — модель обучается на прошлом, тестируется на будущем
SPLIT_TIMESTEP = 34

train_mask = timesteps <= SPLIT_TIMESTEP
test_mask  = timesteps >  SPLIT_TIMESTEP

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train: {X_train.shape[0]:,} транзакций (timestep <= {SPLIT_TIMESTEP})')
print(f'Test : {X_test.shape[0]:,}  транзакций (timestep >  {SPLIT_TIMESTEP})')
print(f'Train illicit rate: {y_train.mean()*100:.1f}%')
print(f'Test  illicit rate: {y_test.mean()*100:.1f}%')

In [ ]:
# ---- 4.2 Масштабирование ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ---- 4.3 Балансировка классов (SMOTE) ----
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'До SMOTE  — illicit: {y_train.sum():,}, licit: {(y_train==0).sum():,}')
print(f'После SMOTE — illicit: {y_train_res.sum():,}, licit: {(y_train_res==0).sum():,}')

## 5. Моделирование

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, average_precision_score, f1_score
)
import time

# ---- 5.1 Определяем модели ----
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=0.1, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=(y_train==0).sum() / y_train.sum(),  # дисбаланс классов
        random_state=42, eval_metric='logloss', verbosity=0
    ),
}

results = {}
trained_models = {}

print('Обучение моделей...')
for name, model in models.items():
    print(f'\n🔄 {name}...', end=' ')
    t0 = time.time()
    
    # Для LR и RF используем SMOTE-данные; для XGBoost — scale_pos_weight
    if name == 'XGBoost':
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train_res, y_train_res)
    
    elapsed = time.time() - t0
    
    # Предсказания
    y_pred      = model.predict(X_test_scaled)
    y_proba     = model.predict_proba(X_test_scaled)[:, 1]
    
    # Метрики
    roc_auc     = roc_auc_score(y_test, y_proba)
    avg_prec    = average_precision_score(y_test, y_proba)
    f1_ill      = f1_score(y_test, y_pred, pos_label=1)
    
    results[name] = {
        'y_pred': y_pred, 'y_proba': y_proba,
        'ROC-AUC': roc_auc, 'PR-AUC': avg_prec, 'F1 (illicit)': f1_ill,
        'time': elapsed
    }
    trained_models[name] = model
    print(f'✅ ROC-AUC={roc_auc:.4f} | PR-AUC={avg_prec:.4f} | F1={f1_ill:.4f} | {elapsed:.1f}s')

In [ ]:
# ---- 5.2 Сводная таблица метрик ----
summary = pd.DataFrame({
    name: {
        'ROC-AUC': f"{r['ROC-AUC']:.4f}",
        'PR-AUC':  f"{r['PR-AUC']:.4f}",
        'F1 (illicit)': f"{r['F1 (illicit)']:.4f}",
        'Время обучения (с)': f"{r['time']:.1f}"
    }
    for name, r in results.items()
}).T

print('\n=== Сравнение моделей ===')
display(summary)

best_model_name = max(results, key=lambda x: results[x]['ROC-AUC'])
print(f'\n🏆 Лучшая модель: {best_model_name} (ROC-AUC = {results[best_model_name]["ROC-AUC"]:.4f})')

In [ ]:
# ---- 5.3 Детальный classification report для лучшей модели ----
print(f'\nDetailed Report: {best_model_name}')
print(classification_report(
    y_test, results[best_model_name]['y_pred'],
    target_names=['Легитимная (0)', 'Подозрительная (1)']
))

In [ ]:
# ---- 5.4 Визуализация: ROC, PR-кривые, Confusion Matrix ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Оценка моделей', fontsize=14, fontweight='bold')

colors_map = {'Logistic Regression': '#3498db', 'Random Forest': '#2ecc71', 'XGBoost': '#e74c3c'}

# ROC curve
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['ROC-AUC']:.3f})", color=colors_map[name], linewidth=2)
axes[0].plot([0,1],[0,1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# PR curve
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r['y_proba'])
    axes[1].plot(rec, prec, label=f"{name} (AP={r['PR-AUC']:.3f})", color=colors_map[name], linewidth=2)
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Confusion Matrix лучшей модели
cm = confusion_matrix(y_test, results[best_model_name]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Легитимная', 'Подозрительная'],
            yticklabels=['Легитимная', 'Подозрительная'])
axes[2].set_title(f'Confusion Matrix\n({best_model_name})')
axes[2].set_ylabel('Истинный класс')
axes[2].set_xlabel('Предсказанный класс')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Тонкая настройка модели (Hyperparameter Tuning)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Настройка XGBoost (как правило лучшей модели)
param_grid = {
    'n_estimators':  [200, 300, 500],
    'max_depth':     [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_base = XGBClassifier(
    scale_pos_weight=(y_train==0).sum() / y_train.sum(),
    random_state=42, eval_metric='logloss', verbosity=0
)

cv = StratifiedKFold(n_splits=5, shuffle=False)  # no shuffle — временные данные

search = RandomizedSearchCV(
    xgb_base, param_grid, n_iter=20,
    scoring='roc_auc', cv=cv, random_state=42, n_jobs=-1, verbose=1
)

search.fit(X_train_scaled, y_train)

best_xgb = search.best_estimator_
print(f'\nЛучшие параметры: {search.best_params_}')
print(f'CV ROC-AUC: {search.best_score_:.4f}')

# Оценка на тесте
y_proba_tuned = best_xgb.predict_proba(X_test_scaled)[:, 1]
y_pred_tuned  = best_xgb.predict(X_test_scaled)
print(f'\nTest ROC-AUC (tuned): {roc_auc_score(y_test, y_proba_tuned):.4f}')
print(f'Test F1 illicit (tuned): {f1_score(y_test, y_pred_tuned):.4f}')

## 7. Анализ стабильности данных (Data Drift Monitoring)

Это ключевое требование курса — анализ изменений в распределении данных по времени.

In [ ]:
from scipy import stats

# ---- 7.1 Population Stability Index (PSI) ----
# PSI — стандартная метрика мониторинга в финансовой отрасли

def compute_psi(expected, actual, buckets=10, eps=1e-6):
    """Compute PSI between two distributions."""
    bins = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    bins[0]  = -np.inf
    bins[-1] =  np.inf
    
    exp_counts = np.histogram(expected, bins=bins)[0]
    act_counts = np.histogram(actual,   bins=bins)[0]
    
    exp_pct = exp_counts / len(expected) + eps
    act_pct = act_counts / len(actual)   + eps
    
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return psi

# Используем первый временной период как baseline
reference_mask = timesteps <= 10
X_ref = X[reference_mask]

# Считаем PSI для каждого временного окна
window_psi = {}
windows = [(11,20), (21,30), (31,40), (41,49)]

top10_feats_idx = [feat_cols.index(f) for f in top_features.head(10).index if f in feat_cols]

for w_start, w_end in windows:
    w_mask = (timesteps >= w_start) & (timesteps <= w_end)
    X_window = X[w_mask]
    psi_vals = [
        compute_psi(X_ref[:, i], X_window[:, i])
        for i in top10_feats_idx
    ]
    window_psi[f'{w_start}-{w_end}'] = np.mean(psi_vals)

print('PSI интерпретация: < 0.1 = стабильно | 0.1-0.25 = умеренный дрейф | > 0.25 = критический дрейф')
print()
for period, psi in window_psi.items():
    status = '✅ Стабильно' if psi < 0.1 else ('⚠️ Умеренный дрейф' if psi < 0.25 else '🚨 Критический дрейф')
    print(f'Период {period}: PSI = {psi:.4f} → {status}')

In [ ]:
# ---- 7.2 Визуализация дрейфа данных ----
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Мониторинг стабильности данных (Data Drift)', fontsize=14, fontweight='bold')

# PSI по периодам
periods = list(window_psi.keys())
psi_values = list(window_psi.values())
bar_colors = ['#2ecc71' if p < 0.1 else '#f39c12' if p < 0.25 else '#e74c3c' for p in psi_values]

axes[0].bar(periods, psi_values, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[0].axhline(0.10, color='orange', linestyle='--', linewidth=1.5, label='Порог умеренного дрейфа (0.10)')
axes[0].axhline(0.25, color='red',    linestyle='--', linewidth=1.5, label='Порог критического дрейфа (0.25)')
axes[0].set_title('PSI (Population Stability Index) по временным периодам')
axes[0].set_xlabel('Период (timestep)')
axes[0].set_ylabel('PSI')
axes[0].legend()

# KS-тест: дрейф отдельных признаков
ks_stats = []
feat_labels = []
X_late = X[timesteps > 34]
for i in top10_feats_idx[:8]:
    ks_stat, p_val = stats.ks_2samp(X_ref[:, i], X_late[:, i])
    ks_stats.append(ks_stat)
    feat_labels.append(feat_cols[i])

ks_colors = ['#e74c3c' if s > 0.1 else '#3498db' for s in ks_stats]
axes[1].barh(feat_labels, ks_stats, color=ks_colors, edgecolor='black', linewidth=0.5)
axes[1].axvline(0.1, color='red', linestyle='--', label='Порог значимого дрейфа (KS=0.1)')
axes[1].set_title('KS-статистика по признакам (ранний период vs поздний период)')
axes[1].set_xlabel('KS Statistics')
axes[1].legend()

plt.tight_layout()
plt.savefig('data_drift.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 7.3 Деградация модели во времени ----
# Как меняется качество модели на разных временных окнах?

model_performance_over_time = []
time_windows = [(1,5),(6,10),(11,15),(16,20),(21,25),(26,30),(31,35),(36,40),(41,45),(46,49)]

for w_start, w_end in time_windows:
    w_mask = (timesteps > SPLIT_TIMESTEP) & (timesteps >= w_start) & (timesteps <= w_end)
    if w_mask.sum() < 10 or y[w_mask].sum() < 2:
        continue
    X_w = scaler.transform(X[w_mask])
    y_w = y[w_mask]
    proba_w = best_xgb.predict_proba(X_w)[:, 1]
    try:
        auc = roc_auc_score(y_w, proba_w)
        model_performance_over_time.append({'period': f'{w_start}-{w_end}', 'ROC-AUC': auc, 'n': w_mask.sum()})
    except:
        pass

perf_df = pd.DataFrame(model_performance_over_time)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(perf_df)), perf_df['ROC-AUC'], marker='o', linewidth=2, color='#3498db', markersize=8)
ax.fill_between(range(len(perf_df)), perf_df['ROC-AUC'], alpha=0.15, color='#3498db')
ax.set_xticks(range(len(perf_df)))
ax.set_xticklabels(perf_df['period'], rotation=30)
ax.set_title('Деградация модели XGBoost во времени (ROC-AUC по временным окнам)', fontweight='bold')
ax.set_ylabel('ROC-AUC')
ax.set_xlabel('Временной период')
ax.set_ylim([0.5, 1.0])
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Минимально приемлемый уровень (0.80)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('model_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Интерпретация модели (SHAP)

In [ ]:
import shap

# SHAP values для XGBoost
print('Вычисление SHAP значений (может занять 1-2 минуты)...')
explainer = shap.TreeExplainer(best_xgb)

# Используем сэмпл для скорости
sample_idx = np.random.choice(len(X_test_scaled), size=min(2000, len(X_test_scaled)), replace=False)
X_shap = X_test_scaled[sample_idx]

shap_values = explainer.shap_values(X_shap)
print('✅ SHAP значения вычислены')

In [ ]:
# ---- 8.1 Summary plot ----
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feat_cols,
    max_display=15,
    show=False
)
plt.title('SHAP Summary Plot — Влияние признаков на предсказание (illicit)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 8.2 Bar plot — Mean |SHAP| ----
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feat_cols,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('Топ-15 признаков по средней важности (SHAP)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Анализ порога классификации (Threshold Analysis)

В задаче AML критично минимизировать пропуски подозрительных транзакций (FN), допуская больше ложных тревог (FP).

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.05)
precision_arr, recall_arr, f1_arr = [], [], []

y_proba_best = best_xgb.predict_proba(X_test_scaled)[:, 1]

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec  = recall_score(y_test, y_pred_t, zero_division=0)
    f1   = f1_score(y_test, y_pred_t, zero_division=0)
    precision_arr.append(prec)
    recall_arr.append(rec)
    f1_arr.append(f1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, precision_arr, 'b-o', markersize=4, label='Precision')
ax.plot(thresholds, recall_arr,    'r-o', markersize=4, label='Recall')
ax.plot(thresholds, f1_arr,        'g-o', markersize=4, label='F1')

best_t_idx = np.argmax(f1_arr)
ax.axvline(thresholds[best_t_idx], color='purple', linestyle='--', 
           label=f'Оптим. порог F1 = {thresholds[best_t_idx]:.2f}')

# AML рекомендуемый порог — низкий для максимального recall
ax.axvline(0.3, color='orange', linestyle=':', linewidth=2,
           label='Рекоменд. для AML (0.30) — макс. recall')

ax.set_xlabel('Порог классификации')
ax.set_ylabel('Метрика')
ax.set_title('Зависимость Precision / Recall / F1 от порога классификации', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'При пороге 0.30 (рекоменд. для AML):')
y_pred_03 = (y_proba_best >= 0.3).astype(int)
print(classification_report(y_test, y_pred_03, target_names=['Легитимная', 'Подозрительная']))

## 10. Итоги и RegTech-выводы

In [ ]:
print('=' * 60)
print('ИТОГОВЫЙ ОТЧЁТ — AML-мониторинг Bitcoin транзакций')
print('=' * 60)

print('\n📊 ДАТАСЕТ')
print(f'  Elliptic Bitcoin Dataset: {len(df):,} транзакций')
print(f'  Размеченных: {len(df_labeled):,} | Illicit: {(df_labeled["label"]==1).sum():,} ({(df_labeled["label"]==1).mean()*100:.1f}%)')
print(f'  49 временных шагов, 166 признаков')

print('\n🤖 ЛУЧШАЯ МОДЕЛЬ')
y_proba_best = best_xgb.predict_proba(X_test_scaled)[:, 1]
print(f'  XGBoost (tuned)')
print(f'  ROC-AUC  : {roc_auc_score(y_test, y_proba_best):.4f}')
print(f'  PR-AUC   : {average_precision_score(y_test, y_proba_best):.4f}')
print(f'  F1 (illicit): {f1_score(y_test, (y_proba_best>=0.3).astype(int)):.4f}  (при пороге 0.3)')

print('\n📈 DATA DRIFT')
for period, psi in window_psi.items():
    status = '✅' if psi < 0.1 else ('⚠️' if psi < 0.25 else '🚨')
    print(f'  {status} Period {period}: PSI = {psi:.4f}')

print('\n🏛️ REGTECH-ИМПЛИКАЦИИ')
print('  1. Модель может автоматически флагировать подозрительные транзакции')
print('     для ручной проверки комплаенс-офицером')
print('  2. Низкий порог (0.30) снижает риск пропуска отмывания денег (FN)')
print('  3. PSI-мониторинг позволяет выявлять изменения в поведении участников')
print('  4. Соответствует требованиям FATF Rec.16 (Travel Rule) по идентификации')
print('  5. Может интегрироваться в систему AML-мониторинга регулятора')
print()
print('Сохранённые файлы:')
saved = ['class_distribution.png','temporal_distribution.png','feature_correlation.png',
         'feature_distributions.png','correlation_heatmap.png','model_evaluation.png',
         'data_drift.png','model_degradation.png','shap_summary.png',
         'shap_importance.png','threshold_analysis.png']
for f in saved:
    print(f'  ✅ {f}')

In [ ]:
# Сохранение обученной модели
import joblib
joblib.dump(best_xgb, 'aml_xgboost_model.pkl')
joblib.dump(scaler,   'aml_scaler.pkl')
print('✅ Модель сохранена: aml_xgboost_model.pkl')
print('✅ Скейлер сохранён: aml_scaler.pkl')